> ✅ **SOLVED VERSION**: every `# TODO` is filled in and every written answer is completed. Numbers quoted in the written answers are *typical*; check them against your own printed results.

# 🏭 LSTM in the Real World: three industry mini-projects

You will **not** write LSTMs from scratch. You will use professional libraries (**Keras/TensorFlow, scikit-learn, pandas, statsmodels**) the way a data scientist does on the job:

| Project | Business problem | Dataset (real → fallback) |
|---|---|---|
| **A. Sentiment analysis** | "Is this customer review positive or negative?" | IMDB reviews → synthetic reviews |
| **B. Next-word prediction** | "Suggest the next word while the user types" | Shakespeare text → synthetic sentences |
| **C. Time-series forecasting** | "What will the value be next month?" | Mauna Loa CO₂ (bundled in `statsmodels`) → synthetic series |

Each project follows the real workflow: **understand the problem → baseline first → LSTM → honest comparison → decision**.
An LSTM is one tool. A good engineer knows when *not* to use it, so that is part of every task.

**How to work:** fill in every `# TODO`. After most tasks, a ✅ **Check** cell must print `✔ passed`. Questions go in the cells marked **Your answer**.
💡 Tip: use a GPU runtime (Colab: *Runtime → Change runtime type → GPU*). If your machine is slow, set `FAST = True` in the setup cell.

## 0. The expert's guide: when to use an LSTM (and when not to)

### What an LSTM is good at
An LSTM reads a sequence **step by step** and keeps a *memory* (cell state) that gates decide to keep, write or erase. It is useful when **order matters** and the information you need can be **several to a few hundred steps back**.

### ✅ Good fit
| Situation | Examples |
|---|---|
| Sequential data where order carries meaning | text, sensor/IoT streams, logs, clickstreams, audio features |
| Short/medium context (tens to a few hundred steps) | next-word suggestion, review sentiment, predictive maintenance windows |
| Multivariate time series with nonlinear interactions and many series | demand across many stores, energy load with weather and calendar |
| Small/medium models that must run on a phone or edge device | keyboard suggestions, wearable activity recognition |
| Streaming data where you carry state forward cheaply | real-time anomaly detection on sensors |

### ❌ Avoid (or at least try something simpler first)
| Situation | Better choice | Why |
|---|---|---|
| **Tabular data** (rows = customers, columns = features) | Gradient boosting (XGBoost / LightGBM), logistic regression | No sequence to exploit. GBMs usually win. |
| Short text classification, plenty of keywords | TF-IDF + logistic regression | Fast, strong, explainable, tiny |
| Long documents (thousands of tokens), large corpora, state-of-the-art NLP | Pre-trained Transformers (BERT family) | LSTMs forget over long ranges and train slowly (no parallelism over time) |
| One short, clean, strongly seasonal series | ETS, SARIMA, Prophet, or just a seasonal-naive baseline | Statistical models are as good with less data and give confidence intervals |
| Very little data (hundreds of rows) | Simple models | LSTMs overfit |
| Needs full explainability / audits | Linear models, trees | LSTM is a black box |
| Large-scale training speed matters | CNNs / Transformers | LSTM steps are sequential |

### 🧭 The professional workflow (you will repeat this three times)
1. **Frame the problem**: what is the input, the output, the cost of errors?
2. **Split correctly**: random for independent samples, **chronological for time series** (no leakage!).
3. **Build a cheap baseline first.** If the baseline is almost as good, ship the baseline.
4. **Train the LSTM** with early stopping; watch train vs validation loss.
5. **Compare honestly**: quality, training time, latency, model size, explainability.
6. **Look at the errors**, not just the metric.

### Quick self-check
Which would you try an LSTM on? *(just think, answers in the final section)*
(a) Predict customer churn from 30 account columns. (b) Detect a failing machine from 1 hour of vibration readings. (c) Classify 20-page legal contracts. (d) Phone keyboard word suggestion.

## Setup

In [ ]:
# If you are on Colab and something is missing:  !pip -q install statsmodels scikit-learn pandas matplotlib
import os, re, time, random
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, f1_score, mean_absolute_error

SEED = 42
keras.utils.set_random_seed(SEED)

FAST = False   # True -> smaller data / fewer epochs for slow computers
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

---
# Project A: Sentiment analysis of customer reviews

### 📘 The business problem
An e-commerce company gets thousands of reviews daily. Management wants an automatic **positive / negative** label to track satisfaction and route angry customers to support.

**Problem type:** text classification (many-to-one). Order matters a little ("not good" ≠ "good"), so an LSTM is plausible. But keyword-based models are strong competitors, so **baseline first**.

**Standard LSTM text pipeline in Keras**
`raw text → TextVectorization (words → integer ids, pad/truncate) → Embedding → (Bidirectional) LSTM → Dropout → Dense(sigmoid)`

**Metrics:** accuracy is fine when classes are balanced; otherwise use precision / recall / **F1**.

In [ ]:
def make_synthetic_reviews(n, seed=0):
    '''Fallback dataset (used only if the real IMDB download is unavailable).'''
    rng = random.Random(seed)
    pos = ["great", "excellent", "amazing", "fantastic", "wonderful", "perfect", "enjoyable", "brilliant"]
    neg = ["terrible", "awful", "boring", "disappointing", "poor", "useless", "horrible", "dreadful"]
    nouns = ["movie", "product", "service", "story", "acting", "delivery", "quality", "design"]
    filler = ["i bought it last week", "we watched it on friday", "it arrived in a plain box",
              "my friend recommended it", "the price was average", "i have used it for a month"]
    rows = []
    for _ in range(n):
        label = int(rng.random() < 0.5)
        main, other = (pos, neg) if label else (neg, pos)
        sents = []
        for _ in range(rng.randint(2, 7)):
            r, noun = rng.random(), rng.choice(nouns)
            if r < 0.40:   sents.append(f"the {noun} was {rng.choice(main)}")
            elif r < 0.55: sents.append(f"the {noun} was not {rng.choice(other)}")        # negation
            elif r < 0.65: sents.append(f"at first the {noun} seemed {rng.choice(other)} but later it was {rng.choice(main)}")
            else:          sents.append(rng.choice(filler))
        if rng.random() < 0.04: label = 1 - label                                          # label noise
        rows.append((". ".join(sents) + ".", label))
    return pd.DataFrame(rows, columns=["text", "label"])

def load_sentiment_data(n_train=8000, n_test=2000):
    n_train, n_test = (2000, 600) if FAST else (n_train, n_test)
    try:
        (x_tr, y_tr), (x_te, y_te) = keras.datasets.imdb.load_data(num_words=10000)
        inv = {v + 3: k for k, v in keras.datasets.imdb.get_word_index().items()}
        decode = lambda seq: " ".join(inv.get(i, "unk") for i in seq if i > 2)
        rng = np.random.default_rng(SEED)
        i_tr = rng.choice(len(x_tr), n_train, replace=False); i_te = rng.choice(len(x_te), n_test, replace=False)
        df_tr = pd.DataFrame({"text": [decode(x_tr[i]) for i in i_tr], "label": y_tr[i_tr]})
        df_te = pd.DataFrame({"text": [decode(x_te[i]) for i in i_te], "label": y_te[i_te]})
        return df_tr, df_te, "IMDB (real data)"
    except Exception as e:
        print("Could not download IMDB -> using synthetic reviews. (", type(e).__name__, ")")
        return make_synthetic_reviews(n_train, 1), make_synthetic_reviews(n_test, 2), "synthetic reviews"

df_train, df_test, SENT_SOURCE = load_sentiment_data()
X_train_text, y_train = df_train["text"].tolist(), df_train["label"].values
X_test_text,  y_test  = df_test["text"].tolist(),  df_test["label"].values
print("Data source:", SENT_SOURCE, "| train:", len(df_train), "| test:", len(df_test))
df_train.head(3)

### ✏️ Task A1: Explore before modelling
1. What is the share of positive reviews in the training set? (Is it balanced?)
2. Compute the review length in words and choose `max_len` = the **95th percentile** of lengths, capped at 200 (longer reviews will be truncated: this controls memory and training time).

In [ ]:
lengths = df_train["text"].str.split().str.len()

# SOLUTION 1: positive share (hint: df_train["label"].mean())
pos_share = float(df_train["label"].mean())
# SOLUTION 2: 95th percentile of `lengths` (np.percentile), converted to int and capped at 200 (min)
max_len = int(min(np.percentile(lengths, 95), 200))

print(f"positive share = {pos_share:.2f} | max_len = {max_len}")
lengths.hist(bins=40); plt.axvline(max_len, color="red"); plt.title("Review length (words)"); plt.show()

In [ ]:
# ✅ Check A1
assert 0 < pos_share < 1 and isinstance(max_len, int) and 5 < max_len <= 200
print("✔ passed")

### ✏️ Task A2: Build the baseline (TF-IDF + Logistic Regression)
This is what many companies actually ship. Use **scikit-learn**: `make_pipeline(TfidfVectorizer(...), LogisticRegression(...))`.
Use unigrams+bigrams (`ngram_range=(1, 2)`), `max_features=20000`, `max_iter=1000`.

In [ ]:
t0 = time.time()
# SOLUTION 1: create the pipeline
baseline = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), max_features=20000),
                         LogisticRegression(max_iter=1000))
baseline.fit(X_train_text, y_train)

# SOLUTION 3: predict labels for X_test_text
y_pred_base = baseline.predict(X_test_text)
base_time = time.time() - t0

print(f"Baseline finished in {base_time:.1f}s")
print(classification_report(y_test, y_pred_base, digits=3))

In [ ]:
# ✅ Check A2
assert f1_score(y_test, y_pred_base) > 0.75, "Baseline F1 should be > 0.75, check your pipeline"
print("✔ passed")

### ✏️ Task A3: Train the LSTM with Keras
First the text is converted to integer sequences (given). Then build:
`Embedding(MAX_TOKENS, 64, mask_zero=True) → Bidirectional(LSTM(64)) → Dropout(0.3) → Dense(1, sigmoid)`

`mask_zero=True` tells the LSTM to **ignore padding**: an important production detail.

Compile with `binary_crossentropy`, optimizer `adam`, metric `accuracy`, and train with an **EarlyStopping** callback (`monitor="val_loss"`, `patience=1`, `restore_best_weights=True`) and `validation_split=0.15`.

In [ ]:
MAX_TOKENS = 10000
vectorize = layers.TextVectorization(max_tokens=MAX_TOKENS, output_sequence_length=max_len)
vectorize.adapt(X_train_text)                                  # learns the vocabulary from TRAIN only
Xtr = vectorize(tf.constant(X_train_text)).numpy()
Xte = vectorize(tf.constant(X_test_text)).numpy()
print("Sequences:", Xtr.shape, "| first review as ids:", Xtr[0][:12])

In [ ]:
# SOLUTION 1: build the model
lstm_model = keras.Sequential([
    keras.Input(shape=(max_len,)),
    layers.Embedding(MAX_TOKENS, 64, mask_zero=True),   # mask_zero -> padding is ignored by the LSTM
    layers.Bidirectional(layers.LSTM(64)),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
])

# SOLUTION 2: compile
lstm_model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

# SOLUTION 3: early stopping callback
early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=1, restore_best_weights=True)

# SOLUTION 4: fit
history = lstm_model.fit(Xtr, y_train, epochs=6, batch_size=64,
                         validation_split=0.15, callbacks=[early_stop], verbose=2)
lstm_model.summary()

In [ ]:
# ✅ Check A3
assert lstm_model.output_shape == (None, 1)
y_prob = lstm_model.predict(Xte, verbose=0).ravel()
y_pred_lstm = (y_prob > 0.5).astype(int)
assert f1_score(y_test, y_pred_lstm) > 0.70, "LSTM F1 too low. Did you train it?"
print("✔ passed | LSTM F1 =", round(f1_score(y_test, y_pred_lstm), 3))

### ✏️ Task A4: Plot the learning curves and diagnose
Plot training vs validation loss from `history.history`. Is the model **overfitting** (training loss keeps falling while validation loss rises)?

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history.history["loss"], marker="o", label="training loss")
plt.plot(history.history["val_loss"], marker="o", label="validation loss")
plt.xlabel("epoch"); plt.ylabel("binary cross-entropy"); plt.title("BiLSTM learning curves")
plt.legend(); plt.show()

**Your diagnosis (2 sentences):** Training loss keeps falling from epoch to epoch while the validation loss reaches its minimum after only 1-3 epochs and then flattens or rises, which is the classic sign of overfitting (the model starts to memorise training reviews). That is exactly why we use EarlyStopping with `restore_best_weights=True`: it stops at the epoch with the best validation loss and keeps those weights.

### ✏️ Task A5: Compare like an engineer
Fill in the comparison table. We measure **latency** = seconds to score the whole test set, and **size** = number of parameters.

In [ ]:
# SOLUTION 1: time the baseline prediction on X_test_text  (use time.time() before/after baseline.predict)
_t = time.time(); baseline.predict(X_test_text); base_latency = time.time() - _t
# SOLUTION 2: time the LSTM prediction on Xte (lstm_model.predict(Xte, verbose=0))
_t = time.time(); lstm_model.predict(Xte, verbose=0); lstm_latency = time.time() - _t
base_params = len(baseline[0].vocabulary_) + 1    # roughly: one weight per feature
lstm_params = lstm_model.count_params()

comparison = pd.DataFrame({
    "F1":              [f1_score(y_test, y_pred_base), f1_score(y_test, y_pred_lstm)],
    "train time (s)":  [base_time, np.nan],
    "latency (s)":     [base_latency, lstm_latency],
    "parameters":      [base_params, lstm_params],
}, index=["TF-IDF + LogReg", "BiLSTM"]).round(3)
comparison

In [ ]:
# ✅ Check A5
assert base_latency is not None and lstm_latency is not None and base_latency > 0
print("✔ passed")

### ✏️ Task A6: Error analysis and probing
Metrics hide *why* a model fails. 
1. Print 5 test reviews that the LSTM got wrong (shortened to 200 characters).
2. Write **6 probe sentences** of your own that test **negation** ("not good"), **sarcasm**, and **mixed opinions**, then see how both models behave.

In [ ]:
# SOLUTION 1: indices where y_pred_lstm != y_test -> print the first 5 texts with true label and predicted label
wrong = np.where(y_pred_lstm != y_test)[0]
print(f"{len(wrong)} wrong out of {len(y_test)}\n")
for i in wrong[:5]:
    print(f"true={y_test[i]} | predicted={y_pred_lstm[i]} (P(pos)={y_prob[i]:.2f}) | {X_test_text[i][:200]!r}\n")

# SOLUTION 2: write 6 probe sentences
probes = [
    "This product is not good at all.",                                   # negation
    "Oh great, it broke after one day. Just what I needed.",              # sarcasm
    "The acting was wonderful but the story was boring and far too long.",  # mixed opinion
    "I wouldn't say it was bad, but I wouldn't recommend it either.",     # double negation / hedging
    "Absolutely fantastic, I loved every single minute!",                 # clear positive
    "Not bad at all, actually quite good.",                               # negation of a negative
]

if probes and probes[0] != "...":
    p_base = baseline.predict_proba(probes)[:, 1]
    p_lstm = lstm_model.predict(vectorize(tf.constant(probes)).numpy(), verbose=0).ravel()
    display(pd.DataFrame({"sentence": probes, "P(positive) baseline": p_base.round(2), "P(positive) LSTM": p_lstm.round(2)}))

### 🧠 Decision A (write 4–6 sentences)
You are the data scientist. The company must score **50 million reviews per day** on cheap servers and wants to explain decisions to support agents. Which model would you ship: the baseline, the LSTM, or something else (e.g. a fine-tuned small Transformer)? Use your table, the errors and probes as evidence.

**Your answer:** I would ship the **TF-IDF + logistic regression baseline** (or a distilled/fine-tuned small Transformer only if a business case justifies the cost). 50 million reviews per day is about 580 reviews per second on average, and the linear model scores a whole test set in a fraction of a second on one CPU core, trains in seconds, is tiny and needs no GPU, whereas the BiLSTM is larger and slower (compare the latency and parameter columns of the table) and, in my results, its F1 is at best similar to the baseline's (check your own table). It is also explainable: every word/bigram has a coefficient, so support agents can see *why* a review was flagged, which an LSTM cannot offer. The probes show that both models handle simple negation, thanks to bigrams in the baseline and word order in the LSTM, but both fail on sarcasm and mixed opinions, so the extra complexity of the LSTM does not buy a different class of mistake. If accuracy on hard cases becomes important, I would add a fine-tuned small Transformer as a second stage only for low-confidence reviews.

---
# Project B: Next-word prediction (keyboard autocomplete)

### 📘 The business problem
A messaging app wants to show the **3 most likely next words** above the keyboard.

**Problem type:** language modelling = "given the last *k* words, predict word *k+1*" (many-to-one, a multi-class classification over the vocabulary).

**Pipeline:** `text → TextVectorization → one long list of word ids → sliding windows (k words → next word) → Embedding → LSTM → Dense(softmax over vocab)`

**Metrics:**
- **Top-1 / Top-3 accuracy** (the user sees 3 suggestions, so top-3 is the product metric)
- **Perplexity** = `exp(cross-entropy)`; lower is better. It is roughly "how many words the model is torn between".

**LSTM or not?** An LSTM is a good fit for a **small, on-device, domain-specific** model. For open-domain, high-quality text generation, large pre-trained Transformers win. And a plain **n-gram table** is a very strong cheap baseline, so we compare.

In [ ]:
def make_synthetic_corpus(n_sentences=6000, seed=0):
    '''Fallback corpus (used only if the Shakespeare download is unavailable).'''
    rng = random.Random(seed)
    subjects = ["the customer", "the manager", "our team", "the driver", "my sister"]
    verbs = {"customer": ["ordered", "returned", "reviewed"], "manager": ["approved", "rejected", "scheduled"],
             "team": ["shipped", "tested", "fixed"], "driver": ["delivered", "loaded", "parked"], "sister": ["bought", "liked", "sent"]}
    objects = ["the package", "a new phone", "the report", "an old laptop", "the invoice", "a small gift"]
    tails = ["on monday", "this morning", "without any delay", "after lunch", "before the deadline"]
    out = []
    for _ in range(n_sentences):
        s = rng.choice(subjects); v = rng.choice(verbs[s.split()[-1]])
        out.append(f"{s} {v} {rng.choice(objects)} {rng.choice(tails)}")
    return "\n".join(out)

def load_corpus():
    try:
        path = keras.utils.get_file("shakespeare.txt", "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt")
        raw = open(path, encoding="utf-8").read()[: (60_000 if FAST else 400_000)]
        return raw, "Shakespeare (real data)"
    except Exception as e:
        print("Could not download Shakespeare -> using synthetic corpus. (", type(e).__name__, ")")
        return make_synthetic_corpus(1500 if FAST else 6000), "synthetic sentences"

raw_text, TEXT_SOURCE = load_corpus()
print("Data source:", TEXT_SOURCE, "|", len(raw_text), "characters")
print(raw_text[:300])

### ✏️ Task B1: Turn text into word ids
The vectorizer is built for you. Your job: inspect it.

In [ ]:
VOCAB = 5000
vectorizer = layers.TextVectorization(max_tokens=VOCAB, standardize="lower_and_strip_punctuation", output_mode="int")
vectorizer.adapt(np.array(raw_text.split("\n")))
ids = vectorizer(tf.constant([raw_text])).numpy()[0]          # the whole corpus as one list of word ids
vocab = vectorizer.get_vocabulary()                           # id -> word  (0 = padding, 1 = [UNK])
word2id = {w: i for i, w in enumerate(vocab)}

print(vocab[2:17])   # skip id 0 (padding) and id 1 ([UNK])

# SOLUTION 2: coverage = share of tokens in `ids` that are NOT the unknown token (id 1)
coverage = float((ids != 1).mean())
print(f"{len(ids)} tokens | vocab size {len(vocab)} | coverage {coverage:.1%}")

In [ ]:
# ✅ Check B1
assert 0.5 < coverage <= 1.0
print("✔ passed")

### ✏️ Task B2: Build the training data with `timeseries_dataset_from_array`
Keras can create all sliding windows for you. For a window of `SEQ` words we need:
- `data = arr[:-1]` (the inputs)
- `targets = arr[SEQ:]` (the word that follows each window)
- `sequence_length = SEQ`

Split the corpus **chronologically**: first 90% train, last 10% validation.

In [ ]:
SEQ = 10
cut = int(0.9 * len(ids))
train_ids, val_ids = ids[:cut], ids[cut:]

def make_ds(arr, shuffle):
    # windows arr[i:i+SEQ] -> target arr[i+SEQ] (the word right after the window)
    return keras.utils.timeseries_dataset_from_array(
        data=arr[:-1], targets=arr[SEQ:], sequence_length=SEQ,
        batch_size=128, shuffle=shuffle, seed=SEED)

train_ds, val_ds = make_ds(train_ids, True), make_ds(val_ids, False)
xb, yb = next(iter(train_ds))
print("batch X:", xb.shape, "| batch y:", yb.shape)
print("window :", " ".join(vocab[i] for i in xb[0].numpy()), "->", vocab[yb[0].numpy()])

In [ ]:
# ✅ Check B2
assert xb.shape[1] == SEQ and len(yb.shape) == 1
# the target must be the word right AFTER the window
arr = train_ids; w = xb[0].numpy(); t = int(yb[0].numpy())
starts = [i for i in range(len(arr) - SEQ) if (arr[i:i+SEQ] == w).all()]
assert any(arr[i + SEQ] == t for i in starts), "targets are misaligned"
print("✔ passed")

### ✏️ Task B3: Model, loss and metrics
`Embedding(VOCAB, 100) → LSTM(128) → Dropout(0.3) → Dense(VOCAB, softmax)`.
Loss: `sparse_categorical_crossentropy` (targets are integers). Metrics: `sparse_categorical_accuracy` and `keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3")`.
Use EarlyStopping on `val_loss` (patience 1, restore best weights), up to 8 epochs.

In [ ]:
# SOLUTION 1: build the model
lm = keras.Sequential([
    keras.Input(shape=(SEQ,)),
    layers.Embedding(VOCAB, 100),
    layers.LSTM(128),
    layers.Dropout(0.3),
    layers.Dense(VOCAB, activation="softmax"),
])

# SOLUTION 2: compile with top-1 and top-3 accuracy
lm.compile(loss="sparse_categorical_crossentropy", optimizer="adam",
           metrics=["sparse_categorical_accuracy",
                    keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3")])

# SOLUTION 3: fit with early stopping
early_b = keras.callbacks.EarlyStopping(monitor="val_loss", patience=1, restore_best_weights=True)
history_b = lm.fit(train_ds, validation_data=val_ds, epochs=3 if FAST else 8,
                   callbacks=[early_b], verbose=2)

In [ ]:
val_loss, val_acc, val_top3 = lm.evaluate(val_ds, verbose=0)
print(f"val top-1 = {val_acc:.3f} | val top-3 = {val_top3:.3f} | perplexity = {np.exp(val_loss):.1f}")
assert val_top3 > 0.10, "Top-3 accuracy is very low. Check the model / data."
print("✔ passed")   # ✅ Check B3

### ✏️ Task B4: Ship it, a `suggest_next` function
Write the final step of the product: given a typed sentence, return the 3 best next words.
Hints: `lm.predict(...)` returns probabilities for every word. Ignore ids 0 and 1 (padding / unknown) and use `np.argsort`.

In [ ]:
def suggest_next(text, k=3):
    words = re.sub(r"[^a-z\s]", "", text.lower()).split()
    seq = [word2id.get(w, 1) for w in words]
    x = keras.utils.pad_sequences([seq], maxlen=SEQ, padding="pre", truncating="pre")        # shape (1, SEQ)
    # SOLUTION: probabilities for every word, ignore padding (0) and unknown (1), take the k best
    probs = lm.predict(x, verbose=0)[0].copy()
    probs[0] = 0.0
    probs[1] = 0.0
    top_ids = np.argsort(probs)[::-1][:k]
    return [vocab[i] for i in top_ids]

seeds = (["the quick", "to be or not to", "my lord"] if "Shakespeare" in TEXT_SOURCE else
         ["the customer ordered", "our team", "the manager approved the report"])
for s in seeds:
    print(f"{s!r:40} ->", suggest_next(s))

In [ ]:
# ✅ Check B4
out = suggest_next(seeds[0])
assert isinstance(out, list) and len(out) == 3 and all(isinstance(w, str) for w in out) and "[UNK]" not in out and "" not in out
print("✔ passed")

### ✏️ Task B5: Beat-the-baseline: the bigram table
A bigram model just counts which word most often follows the previous **one** word. Build it with `defaultdict(Counter)` on `train_ids` and compute its **top-3 accuracy** on the validation windows (the same targets the LSTM was scored on: predict `val_ids[i]` from `val_ids[i-1]` for `i ≥ SEQ`).

In [ ]:
bigram = defaultdict(Counter)
# SOLUTION 1: count (previous word -> next word) pairs in the training ids
for prev, nxt in zip(train_ids[:-1], train_ids[1:]):
    bigram[int(prev)][int(nxt)] += 1

# SOLUTION 2: top-3 accuracy on the same validation targets as the LSTM (val_ids[SEQ:])
hits, total = 0, 0
for i in range(SEQ, len(val_ids)):
    prev, target = int(val_ids[i - 1]), int(val_ids[i])
    top3 = [w for w, _ in bigram[prev].most_common(3)]
    hits += int(target in top3)
    total += 1
bigram_top3 = hits / total
print(f"Bigram top-3 = {bigram_top3:.3f}  vs  LSTM top-3 = {val_top3:.3f}")

In [ ]:
# ✅ Check B5
assert 0 < bigram_top3 < 1
print("✔ passed")

### 🧠 Decision B (write 4–6 sentences)
Your LSTM vs the bigram table: how large is the gap? Think about **model size on a phone**, **latency per keystroke**, **updating the model weekly with new slang**, and **privacy**. What would you ship for (1) a keyboard on cheap phones and (2) a "write my email" assistant?

**Your answer:** The LSTM usually beats the bigram table in top-3 accuracy, but the gap is moderate (compare the two numbers printed above): with only the last word a bigram table already predicts many common continuations ("my" -> "lord"), and the LSTM adds context from the previous 10 words. For (1) a keyboard on **cheap phones** I would ship the n-gram/bigram table (or a very small quantised GRU): it is a few hundred KB, a lookup costs microseconds per keystroke, it can be updated every week by simply adding new counts (new slang appears immediately, no retraining) and it can be built and updated on the device, so the user's text never leaves the phone (privacy). The LSTM must be retrained and re-shipped for every update and costs more memory and battery. For (2) a **"write my email" assistant** I would use a large pre-trained Transformer language model: it needs long context, fluent multi-sentence generation and world knowledge, which neither a bigram table nor a small LSTM can provide, and latency/size matter less because it runs on a server or a strong device.

---
# Project C: Time-series forecasting

### 📘 The business problem
An analyst must forecast next month's value of a metric (energy demand, sales, atmospheric CO₂ ...) to plan capacity.

**Problem type:** supervised learning on a *window*: "the last 24 months → next month".

**Rules of the trade (these decide whether your model is honest or useless):**
1. **Chronological split.** The test set is the *future*. Never shuffle before splitting.
2. **Fit scalers on the training part only**: otherwise you leak the future.
3. **Always compare with naive baselines**: "same as last month" and "same as 12 months ago".
4. **Trends are dangerous for neural networks.** LSTM outputs are squashed by `tanh`; they struggle to predict values *outside the range seen in training*. Experts often model the **change** (differences) instead of the level.

**LSTM or not?** For a *single short seasonal series*, ETS / SARIMA / Prophet or even linear regression is usually just as good. LSTMs earn their place with **many series, multiple input variables, nonlinear effects**. We'll see for ourselves.

In [ ]:
def load_series():
    try:
        import statsmodels.api as sm
        co2 = sm.datasets.co2.load_pandas().data["co2"]                 # weekly, bundled with statsmodels (no internet needed)
        s = co2.resample("MS").mean().interpolate()                     # monthly mean, fill small gaps
        return s, "Mauna Loa atmospheric CO2, ppm (real data)"
    except Exception as e:
        print("statsmodels data unavailable -> synthetic series. (", type(e).__name__, ")")
        t = np.arange(526); rng = np.random.default_rng(0)
        s = pd.Series(315 + 0.12 * t + 3 * np.sin(2 * np.pi * t / 12) + rng.normal(0, 0.3, len(t)),
                      index=pd.date_range("1958-03-01", periods=len(t), freq="MS"))
        return s, "synthetic trend + seasonality"

series, TS_SOURCE = load_series()
print("Data source:", TS_SOURCE, "| points:", len(series))
series.plot(figsize=(10, 3), title=TS_SOURCE); plt.show()

### ✏️ Task C1: Chronological split + naive baselines
- `train_end` = index where the last 20% of the series starts (`int(0.8 * len(series))`).
- Compute the **MAE** (`mean_absolute_error`) on the test period of:
  - **naive**: predict `y[t] = y[t-1]`
  - **seasonal naive**: predict `y[t] = y[t-12]`

In [ ]:
vals = series.values.astype("float32")
n = len(vals)
WINDOW = 24

# SOLUTION 1: index where the test period (last 20%) starts
train_end = int(0.8 * n)
# SOLUTION 2: test targets and the two naive predictions for indices train_end ... n-1
y_true = vals[train_end:]
pred_naive = vals[train_end - 1:n - 1]            # y[t] = y[t-1]
pred_seasonal = vals[train_end - 12:n - 12]       # y[t] = y[t-12]

mae = {"naive (last month)": mean_absolute_error(y_true, pred_naive),
       "seasonal naive (12 months ago)": mean_absolute_error(y_true, pred_seasonal)}
print({k: round(v, 3) for k, v in mae.items()})

In [ ]:
# ✅ Check C1
assert train_end == int(0.8 * n) and len(y_true) == len(pred_naive) == len(pred_seasonal) == n - train_end
print("✔ passed")

### ✏️ Task C2: First attempt: LSTM on the raw (scaled) series
1. Scale with `StandardScaler` **fitted on the training part only**.
2. Windows are built for you (`make_xy`, which uses `timeseries_dataset_from_array`). Select training windows (target index `< train_end`) and test windows with a boolean mask.
3. Model: `LSTM(32) → Dense(1)`, `mse` loss, `adam`, up to 40 epochs, `EarlyStopping(patience=5, restore_best_weights=True)`, `validation_split=0.15` (Keras takes the **last** 15%, which is chronological: good).
4. Inverse-transform the predictions and compute test MAE.

In [ ]:
def make_xy(values, window):
    '''values: (n, 1) array -> X (N, window, 1), y (N, 1), where y[j] follows X[j].'''
    ds = keras.utils.timeseries_dataset_from_array(values[:-1], values[window:], sequence_length=window, batch_size=len(values))
    X, y = next(iter(ds))
    return X.numpy(), y.numpy()

EPOCHS_TS = 8 if FAST else 40
# SOLUTION 1: scaler fitted on the TRAINING part only (no leakage from the future)
scaler = StandardScaler().fit(vals[:train_end].reshape(-1, 1))
scaled = scaler.transform(vals.reshape(-1, 1))

X, y = make_xy(scaled, WINDOW)
target_idx = np.arange(len(X)) + WINDOW          # position in the original series of each target
train_mask = target_idx < train_end

# SOLUTION 2: chronological split with the mask
X_tr, y_tr = X[train_mask], y[train_mask]
X_te, y_te = X[~train_mask], y[~train_mask]

# SOLUTION 3: build + compile + fit the model
raw_model = keras.Sequential([
    keras.Input(shape=(WINDOW, 1)),
    layers.LSTM(32),
    layers.Dense(1),
])
raw_model.compile(loss="mse", optimizer="adam")
raw_model.fit(X_tr, y_tr, epochs=EPOCHS_TS, validation_split=0.15, verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])

# SOLUTION 4: predict the test windows and go back to the original units (ppm)
pred_lstm_raw = scaler.inverse_transform(raw_model.predict(X_te, verbose=0)).ravel()
mae["LSTM on raw series"] = mean_absolute_error(y_true, pred_lstm_raw)
print({k: round(v, 3) for k, v in mae.items()})

plt.figure(figsize=(10, 3)); plt.plot(y_true, label="actual"); plt.plot(pred_lstm_raw, label="LSTM raw")
plt.legend(); plt.title("Test period: LSTM on raw values"); plt.show()

In [ ]:
# ✅ Check C2
assert len(pred_lstm_raw) == len(y_true) and X_tr.shape[1:] == (WINDOW, 1)
print("✔ passed")

**Question C2:** compare the LSTM's MAE with the naive baselines and look at the plot. Where does the LSTM go wrong (level? trend? season?) and why do you think so?

**Your answer:** On this trending series the raw-value LSTM is usually **worse than both naive baselines**. The plot shows that the seasonal wiggle is learned reasonably well, but the *level* is wrong: the test period contains CO2 values higher than anything seen in training, and an LSTM (tanh-squashed, trained on a scaled range) cannot extrapolate beyond the range it was trained on, so its predictions lag below the actual curve and the error grows towards the end (a trend problem, not a season problem). The naive baselines do not have this problem because they always start from the latest real observation.

### ✏️ Task C3: The expert fix, forecast the *change*, not the level
1. Compute first differences `d = series.diff()` (use pandas) and drop the first NaN → `d_vals`.
2. Scale with a `StandardScaler` fitted on the training differences only (`d_vals[:train_end - 1]`).
3. Build windows with `make_xy` (the target of sample *j* is the change at series position `j + WINDOW + 1`).
4. Train the **same** LSTM architecture on the differences.
5. Convert back: `prediction[t] = vals[t-1] + predicted_change[t]`.

In [ ]:
# SOLUTION 1: first differences (n-1 values): d[k] = y[k+1] - y[k]
d_vals = series.diff().iloc[1:].values.astype("float32").reshape(-1, 1)
# SOLUTION 2: scaler fitted on the training differences only
d_scaler = StandardScaler().fit(d_vals[:train_end - 1])
d_scaled = d_scaler.transform(d_vals)

Xd, yd = make_xy(d_scaled, WINDOW)
d_target_idx = np.arange(len(Xd)) + WINDOW + 1   # position in the ORIGINAL series
d_train_mask = d_target_idx < train_end
# SOLUTION 3: split into train / test windows
Xd_tr, yd_tr = Xd[d_train_mask], yd[d_train_mask]
Xd_te = Xd[~d_train_mask]

# SOLUTION 4: same model and settings as in C2, trained on the differences
diff_model = keras.Sequential([
    keras.Input(shape=(WINDOW, 1)),
    layers.LSTM(32),
    layers.Dense(1),
])
diff_model.compile(loss="mse", optimizer="adam")
diff_model.fit(Xd_tr, yd_tr, epochs=EPOCHS_TS, validation_split=0.15, verbose=0,
               callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])

# SOLUTION 5: predicted change (back to ppm) + previous true value = predicted level
pred_change = d_scaler.inverse_transform(diff_model.predict(Xd_te, verbose=0)).ravel()
pred_lstm_diff = vals[train_end - 1:n - 1] + pred_change
mae["LSTM on differences"] = mean_absolute_error(y_true, pred_lstm_diff)
print({k: round(v, 3) for k, v in mae.items()})

In [ ]:
# ✅ Check C3
assert len(pred_lstm_diff) == len(y_true)
assert mae["LSTM on differences"] < mae["LSTM on raw series"], "Differencing should help the LSTM on this trending series"
print("✔ passed")

### ✏️ Task C4: The honest challenger, Ridge regression on the same windows
Fit scikit-learn `Ridge(alpha=1.0)` on the **flattened** difference windows (`Xd_tr.reshape(len(Xd_tr), -1)`, `yd_tr.ravel()`), predict the test windows, and convert back exactly as in C3.

In [ ]:
# SOLUTION: Ridge regression on the flattened difference windows
ridge = Ridge(alpha=1.0).fit(Xd_tr.reshape(len(Xd_tr), -1), yd_tr.ravel())
ridge_change = ridge.predict(Xd_te.reshape(len(Xd_te), -1)).reshape(-1, 1)
pred_ridge = vals[train_end - 1:n - 1] + d_scaler.inverse_transform(ridge_change).ravel()

mae["Ridge on differences"] = mean_absolute_error(y_true, pred_ridge)
result = pd.Series(mae).sort_values().round(3).to_frame("test MAE")
display(result)

plt.figure(figsize=(11, 3.5))
plt.plot(y_true, "k", label="actual"); plt.plot(pred_naive, label="naive", alpha=.6)
plt.plot(pred_lstm_diff, label="LSTM (diff)"); plt.plot(pred_ridge, label="Ridge (diff)", alpha=.8)
plt.legend(); plt.title("Test-period forecasts (one month ahead)"); plt.show()

In [ ]:
# ✅ Check C4
assert len(pred_ridge) == len(y_true)
print("✔ passed")

### 🧠 Decision C (write 4–6 sentences)
Look at the table. 
1. Which model is best, and by how much does the LSTM beat (or lose to) the naive baseline and Ridge?
2. If this were *one* series with 500 points, would you deploy an LSTM? 
3. Describe a time-series project where you **would** choose an LSTM (hint: many series, extra input variables, nonlinear patterns, lots of data).

**Your answer:** 1. Look at the sorted table (lowest MAE first): once the LSTM models the *change* it clearly beats the LSTM on raw values and the plain naive baseline, but a simple Ridge regression on the same difference windows is typically just as good, often slightly better, than the LSTM, and the seasonal naive forecast is also very hard to beat for a clean seasonal series. So the LSTM does not win by a meaningful margin here (check the exact differences in your table).
2. For a single series with about 500 points I would **not** deploy an LSTM: Ridge/ETS/SARIMA are as accurate, train instantly, are explainable and give confidence intervals, while the LSTM overfits easily with so little data and is much more expensive to maintain.
3. I would choose an LSTM for a project such as forecasting hourly electricity demand for thousands of smart meters or daily sales of thousands of products: one *global* model is trained on many series together with extra inputs (weather, holidays, promotions, price), there is a lot of data, and the nonlinear interactions between those covariates and the history are what an LSTM can learn and a per-series linear/statistical model cannot.

### 🎁 Optional bonus
- Forecast **12 months ahead** at once: change the targets to the next 12 values and use `Dense(12)`. How fast does the error grow compared to the naive baseline?
- Add a second input feature (e.g. the month number as `sin`/`cos`) and check whether the LSTM improves.

---
# Final section: Be the consultant

For each scenario, decide **LSTM / not LSTM**, name what you would try **first**, and justify in 2–3 sentences. Write your answers in the table.

| # | Scenario | Your decision & first thing to try |
|---|---|---|
| 1 | Predict which of 1 million bank customers will churn from 30 account columns (age, balance, products ...) | **Not LSTM.** Tabular data, no sequence. First try logistic regression, then gradient boosting (XGBoost / LightGBM) with SHAP for explanations; GBMs usually win on tabular churn data and are easy to audit. |
| 2 | Detect failing wind turbines from 1 hour of 50 Hz vibration + temperature readings | **Yes, but not on raw samples.** 1 h at 50 Hz = 180 000 steps per sensor, too long for a raw LSTM. First try a baseline on window features (RMS, FFT bands) with gradient boosting, then an LSTM/GRU or 1-D CNN on short windows or down-sampled data; streaming state makes LSTMs good for real-time alarms. |
| 3 | Classify 20-page legal contracts into 12 categories | **Not LSTM.** Long documents (thousands of tokens): first a TF-IDF + logistic regression baseline, then a pre-trained Transformer (long-context model, or chunk the contract and aggregate chunk predictions). |
| 4 | Phone keyboard word suggestion that works offline | **Yes (small LSTM/GRU).** Small, offline, on-device model. First try an n-gram / bigram table as the baseline, then a small quantised GRU/LSTM if it adds real accuracy. |
| 5 | Forecast daily demand for 5000 stores × 200 products with promotions, holidays and weather | **Yes (one global deep model).** Many related series plus covariates (promotions, holidays, weather) suit an LSTM/GRU or other deep global model. First try seasonal-naive and a LightGBM with lag and calendar features as baselines. |
| 6 | Forecast the monthly revenue of one small shop (3 years of data) | **Not LSTM.** One short series (36 points): start with seasonal naive, then ETS / SARIMA / Prophet; a neural network would overfit. |
| 7 | Product-review sentiment for an app with 10 000 reviews and a tight latency budget | **Probably not LSTM.** 10 000 reviews and a tight latency budget: TF-IDF + logistic regression first (fast, explainable); use an LSTM or distilled Transformer only if it is clearly better on a validation set. |

**Sample reasoning to guide you (read after you answered):** tabular → boosting (1); sensor sequences → LSTM / 1-D CNN, yes (2); long documents → Transformer (3); small on-device model → yes, LSTM/small GRU (4); many series + covariates → yes, LSTM or other deep model with covariates (5); tiny single series → seasonal-naive / ETS (6); baseline TF-IDF first, LSTM only if it clearly beats it (7).

---
🎉 **Well done!** Before submitting: *Kernel → Restart & Run All*, and make sure every ✅ Check prints `✔ passed`.

**Quick self-check from Section 0:** (a) churn from 30 account columns: **no** (tabular, use gradient boosting); (b) failing machine from 1 hour of vibration readings: **yes** (sensor sequence, LSTM / 1-D CNN on windows); (c) 20-page legal contracts: **no** (long documents, use Transformers); (d) phone keyboard word suggestion: **yes** (small on-device sequence model).